#### 10. Test-year analysis

A closer look at 2024 with the final models of notebook 09, loaded from `task2/models/`. Every forecast of the test year is written once to `task2/data/test_predictions/`, as the probabilities of exceeding the eight class edges, so later figures need no model. The notebook then breaks the scores down by month, hour, season and station, puts confidence intervals on the differences between forecasts, extends the calibration check to more thresholds, follows the skill across thresholds, shows one storm, and measures how much the final models rely on each tier. Rows of the test year all have weight 1, so plain averages are used throughout.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../code")
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import array_to_vector

import config as C
import evaluate as E
import mapper as M
import baselines as B

spark = C.get_spark("task2-test-analysis")

final = {kind: PipelineModel.load(str(C.MODELS / kind.lower())) for kind in ("LR", "RF")}
mp = M.load(C.MODELS / "mapper.json")
FEATURES = final["RF"].stages[0].getInputCols()
TIERS = sorted({c.split("_")[0] for c in FEATURES})
NAMES = ["constant", "climatology", "persistence", "LR", "RF"]
KEYS = ["station_id", "slot", "ts", "y_mm", "label", "regime", "past30_mm"]
EDGES = C.BIN_EDGES
PRED = C.DATA / "test_predictions"
CLIM_M = 3000
print(len(FEATURES), "features in", "+".join(t.upper() for t in TIERS))

58 features in T0+T1+T2+T3


**Test predictions**

The baselines are refitted on 2017–2023 exactly as in notebook 09. Each forecast is kept as the array P(Y > e) at the eight class edges e, which holds the same information as the nine class probabilities.

In [2]:
fit = (spark.read.parquet(str(C.ASSEMBLED / "train_full"))
             .unionByName(spark.read.parquet(str(C.ASSEMBLED / "val"))))
test = spark.read.parquet(str(C.ASSEMBLED / "test")).select(*KEYS, *FEATURES)

shares = fit.groupBy("label").agg(F.sum("w").alias("sw")).toPandas().set_index("label").sw.sort_index()
shares = shares / shares.sum()
clim = B.fit_climatology(fit, CLIM_M)
pers = B.fit_persistence(fit)


def with_forecast(df, name, predicted):
    """Add one forecast to df as the array of P(Y > e) at the class edges."""
    ex = E.add_exceedance(predicted)
    return ex.select(*df.columns, F.array(*[F.col(E.exc_col(e)).cast("float") for e in EDGES]).alias(name))


pred = test
pred = with_forecast(pred, "constant", pred.withColumn("probability", array_to_vector(F.array(*[F.lit(float(p)) for p in shares]))))
pred = with_forecast(pred, "climatology", B.predict_climatology(clim, pred))
pred = with_forecast(pred, "persistence", B.predict_persistence(pers, pred))
for kind, model in final.items():
    pred = with_forecast(pred, kind, model.transform(pred))
pred.select(*KEYS, *NAMES).write.mode("overwrite").parquet(str(PRED))

obs = F.array(*[(F.col("label") >= j + 1).cast("float") for j in range(len(EDGES))])
pred = (spark.read.parquet(str(PRED))
             .withColumn("month", F.month("ts")).withColumn("hour", F.hour("ts")).withColumn("day", F.to_date("ts"))
             .withColumn("season", F.when(F.col("month").isin(12, 1, 2, 3), "NE monsoon")
                                    .when(F.col("month").isin(6, 7, 8, 9), "SW monsoon").otherwise("inter-monsoon")))
for name in NAMES:
    pred = pred.withColumn(f"rps_{name}", F.aggregate(F.zip_with(F.col(name), obs, lambda p, o: (p - o) * (p - o)),
                                                      F.lit(0.0), lambda acc, x: acc + x))
pred = pred.cache()
print(f"{pred.count():,} test rows")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


6,374,476 test rows


**Scores and skill**

RPS of each forecast overall and by regime, and the skill score against climatology and against persistence, 1 − RPS / RPS of the reference: the share of the reference's error that the forecast removes.

In [3]:
def rps_by(*keys):
    return pred.groupBy(*keys).agg(F.count("*").alias("n"), F.avg((F.col("label") > 0).cast("double")).alias("rain share"),
                                   *[F.avg(f"rps_{m}").alias(m) for m in NAMES]).toPandas()


overall = pd.concat([rps_by().assign(subset="all"), rps_by("regime").rename(columns={"regime": "subset"})]).set_index("subset")
skill = pd.concat({f"RPSS vs {ref}": 1 - overall[NAMES].div(overall[ref], axis=0) for ref in ("climatology", "persistence")}, axis=1)
overall.to_csv(C.RESULTS / "test_rps_by_regime.csv")
pd.concat([overall[NAMES], skill[[("RPSS vs climatology", m) for m in ("LR", "RF")] + [("RPSS vs persistence", m) for m in ("LR", "RF")]]], axis=1).round(4)

,constant,climatology,persistence,LR,RF,"(RPSS vs climatology, LR)","(RPSS vs climatology, RF)","(RPSS vs persistence, LR)","(RPSS vs persistence, RF)"
subset,,,,,,,,,
all,0.1903,0.1864,0.1246,0.0922,0.0887,0.5054,0.5241,0.2603,0.2882
dry_now,0.0708,0.0710,0.0679,0.0544,0.0530,0.2342,0.2532,0.1991,0.2190
wet_now,1.9499,1.8859,0.9580,0.6487,0.6140,0.6560,0.6744,0.3229,0.3591
unknown,0.1622,0.1580,0.1622,0.0920,0.0880,0.4177,0.4430,0.4327,0.4574


**Confidence intervals**

Rows close in time share the same weather, so they are resampled in whole days: 2,000 bootstrap samples of the 366 days of 2024 give a 95 % interval for each RPS and for the differences between forecasts. A difference whose interval excludes zero is not an accident of this particular year's weather.

In [4]:
daily = pred.groupBy("day").agg(F.count("*").alias("n"), *[F.sum(f"rps_{m}").alias(m) for m in NAMES]).toPandas()
rng = np.random.default_rng(C.SEED)
idx = rng.integers(0, len(daily), size=(2000, len(daily)))
n = daily["n"].to_numpy()[idx].sum(axis=1)
boot = {m: daily[m].to_numpy()[idx].sum(axis=1) / n for m in NAMES}
pairs = {m: boot[m] for m in NAMES}
pairs.update({"RF − LR": boot["RF"] - boot["LR"], "RF − persistence": boot["RF"] - boot["persistence"],
              "LR − persistence": boot["LR"] - boot["persistence"], "RF − climatology": boot["RF"] - boot["climatology"]})
ci = pd.DataFrame({k: {"estimate": v.mean(), "2.5 %": np.percentile(v, 2.5), "97.5 %": np.percentile(v, 97.5),
                       "share below 0": (v < 0).mean()} for k, v in pairs.items()}).T
ci.to_csv(C.RESULTS / "test_bootstrap.csv")
ci.round(5)

,estimate,2.5 %,97.5 %,share below 0
constant,0.19023,0.16457,0.21782,0.0
climatology,0.18631,0.16173,0.21354,0.0
persistence,0.12459,0.11079,0.13913,0.0
LR,0.09217,0.08245,0.10245,0.0
RF,0.08870,0.07943,0.09841,0.0
RF − LR,-0.00347,-0.00418,-0.00286,1.0
RF − persistence,-0.03589,-0.04102,-0.03117,1.0
LR − persistence,-0.03242,-0.03688,-0.02812,1.0
RF − climatology,-0.09762,-0.11699,-0.08019,1.0


**By month, hour and season**

Where in the year and the day the forecasts are weakest. The rain share is the fraction of rows with rain in the next 30 minutes.

In [5]:
by_month = rps_by("month").sort_values("month").set_index("month")
by_hour = rps_by("hour").sort_values("hour").set_index("hour")
by_season = rps_by("season").set_index("season")
for name, table in [("month", by_month), ("hour", by_hour), ("season", by_season)]:
    table.assign(**{"RPSS RF vs climatology": 1 - table["RF"] / table["climatology"]}).to_csv(C.RESULTS / f"test_rps_by_{name}.csv")
by_season.assign(**{"RPSS RF vs climatology": 1 - by_season["RF"] / by_season["climatology"],
                    "RPSS RF vs persistence": 1 - by_season["RF"] / by_season["persistence"]}).round(4)

,n,rain share,constant,climatology,persistence,LR,RF,RPSS RF vs climatology,RPSS RF vs persistence
season,,,,,,,,,
inter-monsoon,2119046,0.0676,0.2086,0.2037,0.1331,0.0966,0.0935,0.5409,0.2976
NE monsoon,2159890,0.0738,0.2117,0.2067,0.1405,0.1058,0.1018,0.5075,0.2750
SW monsoon,2095540,0.0491,0.1497,0.1478,0.0997,0.0736,0.0703,0.5245,0.2947


In [6]:
by_month.assign(**{"RPSS RF vs climatology": 1 - by_month["RF"] / by_month["climatology"]}).round(4)

,n,rain share,constant,climatology,persistence,LR,RF,RPSS RF vs climatology
month,,,,,,,,
1,552686,0.1429,0.3987,0.3885,0.2363,0.1740,0.1659,0.5731
2,514001,0.0412,0.1235,0.1200,0.0996,0.0778,0.0755,0.3710
3,549042,0.0498,0.1472,0.1432,0.1101,0.0847,0.0817,0.4296
4,538660,0.0534,0.1692,0.1659,0.1137,0.0828,0.0802,0.5163
5,547478,0.0610,0.1907,0.1890,0.1276,0.0942,0.0911,0.5180
6,531416,0.0701,0.2235,0.2197,0.1352,0.0996,0.0951,0.5673
7,545442,0.0290,0.0788,0.0768,0.0571,0.0427,0.0404,0.4734
8,498966,0.0402,0.1167,0.1154,0.0967,0.0734,0.0712,0.3830
9,519716,0.0573,0.1803,0.1799,0.1109,0.0797,0.0754,0.5811


In [7]:
by_hour.assign(**{"RPSS RF vs climatology": 1 - by_hour["RF"] / by_hour["climatology"]}).round(4)

,n,rain share,constant,climatology,persistence,LR,RF,RPSS RF vs climatology
hour,,,,,,,,
0,265751,0.0222,0.0540,0.0503,0.0286,0.0222,0.0217,0.5691
1,264904,0.0240,0.0553,0.0519,0.0410,0.0334,0.0318,0.3872
2,264591,0.0318,0.0928,0.0907,0.0670,0.0497,0.0487,0.4633
3,264820,0.0451,0.1535,0.1522,0.0949,0.0610,0.0585,0.6157
4,264816,0.0507,0.1728,0.1720,0.0831,0.0557,0.0541,0.6855
5,264883,0.0540,0.1616,0.1606,0.0867,0.0639,0.0603,0.6242
6,265072,0.0573,0.1707,0.1692,0.0923,0.0668,0.0637,0.6237
7,264653,0.0652,0.2011,0.1995,0.1224,0.0891,0.0841,0.5781
8,265437,0.0768,0.2294,0.2284,0.1312,0.0935,0.0897,0.6073


**By station**

RPS per station with the station's position, for a map of where the forecasts work best.

In [8]:
stations = pd.read_csv(C.STATION_STATIC)[["station_id", "location_longitude", "location_latitude"]]
by_station = rps_by("station_id").merge(stations, on="station_id", how="left")
by_station["RPSS RF vs climatology"] = 1 - by_station["RF"] / by_station["climatology"]
by_station["RPSS RF vs persistence"] = 1 - by_station["RF"] / by_station["persistence"]
by_station.to_csv(C.RESULTS / "test_rps_by_station.csv", index=False)
by_station[["RPSS RF vs climatology", "RPSS RF vs persistence"]].describe().round(3)

,RPSS RF vs climatology,RPSS RF vs persistence
count,70.000,70.000
mean,0.519,0.279
std,0.047,0.063
min,0.237,-0.079
25%,0.515,0.264
50%,0.529,0.293
75%,0.537,0.311
max,0.655,0.368


**Reliability and Brier decomposition**

For P(Y > θ) at four class edges, forecasts are grouped into ten equal-width bins and compared with the observed frequency. The Brier score splits into reliability (calibration error, lower is better), resolution (how far the bins' outcomes differ from the overall rate, higher is better) and uncertainty (the event's own variance, the same for every forecast): BS ≈ REL − RES + UNC.

In [9]:
rel_rows, decomposition = [], []
for theta in [0.0, 1.0, 5.0, 20.0]:
    j = EDGES.index(theta)
    o = (F.col("label") >= j + 1).cast("double")
    for name in ["climatology", "persistence", "LR", "RF"]:
        p = F.col(name)[j]
        bins = (pred.groupBy(F.least(F.floor(p * 10), F.lit(9)).cast("int").alias("bin"))
                    .agg(F.count("*").alias("n"), F.avg(p).alias("mean_forecast"), F.avg(o).alias("observed"))
                    .toPandas().sort_values("bin"))
        rel_rows.append(bins.assign(threshold=theta, forecast=name))
        N, obar = bins.n.sum(), (bins.n * bins.observed).sum() / bins.n.sum()
        decomposition.append({"threshold": theta, "forecast": name,
                              "reliability": (bins.n * (bins.mean_forecast - bins.observed) ** 2).sum() / N,
                              "resolution": (bins.n * (bins.observed - obar) ** 2).sum() / N,
                              "uncertainty": obar * (1 - obar)})
pd.concat(rel_rows).to_csv(C.RESULTS / "test_reliability.csv", index=False)
decomposition = pd.DataFrame(decomposition).set_index(["threshold", "forecast"])
decomposition.to_csv(C.RESULTS / "test_brier_decomposition.csv")
decomposition.round(5)

reliability  resolution  uncertainty
threshold forecast                                         
0.0       climatology      0.00005     0.00106      0.05956
          persistence      0.00001     0.02406      0.05956
          LR               0.00002     0.03329      0.05956
          RF               0.00002     0.03449      0.05956
1.0       climatology      0.00000     0.00001      0.02435
          persistence      0.00001     0.00723      0.02435
          LR               0.00000     0.01157      0.02435
          RF               0.00001     0.01209      0.02435
5.0       climatology      0.00000     0.00000      0.00825
          persistence      0.00000     0.00116      0.00825
          LR               0.00000     0.00268      0.00825
          RF               0.00000     0.00277      0.00825
20.0      climatology      0.00000     0.00000      0.00103
          persistence      0.00000     0.00001      0.00103
          LR               0.00000     0.00014      0.00103
          RF               0.00000     0.00013      0.00103

**Skill across thresholds**

The task asks for P(Y > θ) at any θ. Through the threshold mapper, the Brier skill score against climatology is traced from 0 to 30 mm, including thresholds that fall inside a class. Above about 30 mm a year holds too few events for a stable score, which the event count shows.

In [10]:
def exceed_at(name, theta):
    """P(Y > theta) from the edge array of one forecast, splitting the class that contains theta (mapper)."""
    k, q = M.locate(mp, theta)
    upper = F.col(name)[k] if k < len(EDGES) else F.lit(0.0)
    return upper + F.lit(q) * (F.col(name)[k - 1] - upper)


curve = []
for theta in [0.0, 0.2, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0, 15.0, 20.0, 25.0, 30.0]:
    o = (F.col("y_mm") > theta).cast("double")
    r = pred.agg(F.sum(o).alias("events"), F.avg(o).alias("event rate"),
                 *[F.avg((exceed_at(m, theta) - o) ** 2).alias(m) for m in ["climatology", "persistence", "LR", "RF"]]).first().asDict()
    curve.append({"threshold": theta, "events": int(r["events"]), "event rate": r["event rate"],
                  **{f"BSS {m}": 1 - r[m] / r["climatology"] for m in ["persistence", "LR", "RF"]}})
curve = pd.DataFrame(curve).set_index("threshold")
curve.to_csv(C.RESULTS / "test_skill_by_threshold.csv")
curve.round(4)

,events,event rate,BSS persistence,BSS LR,BSS RF
threshold,,,,,
0.0,405430,0.0636,0.3869,0.5515,0.5711
0.2,280601,0.0440,0.3719,0.5428,0.5634
0.5,231071,0.0362,0.3469,0.5235,0.5437
1.0,159182,0.0250,0.2915,0.4764,0.4966
2.0,105331,0.0165,0.2270,0.4171,0.4322
3.0,79391,0.0125,0.1878,0.3797,0.3900
5.0,53018,0.0083,0.1446,0.3333,0.3427
7.5,35330,0.0055,0.1090,0.2917,0.2951
10.0,24273,0.0038,0.0837,0.2603,0.2603


**A storm in 2024**

The three hours around the largest 30-minute total of the test year at its station: what fell, and what each forecast said beforehand.

In [11]:
peak = pred.orderBy(F.desc("y_mm")).select("station_id", "slot", "ts", "y_mm").first()
storm = (pred.filter((F.col("station_id") == peak.station_id) & F.col("slot").between(peak.slot - 24, peak.slot + 12))
             .orderBy("slot")
             .select(F.date_format("ts", "yyyy-MM-dd HH:mm").alias("time"), "past30_mm", "y_mm",
                     *[F.col(m)[0].alias(f"P(Y>0) {m}") for m in ["persistence", "LR", "RF"]],
                     *[F.col(m)[5].alias(f"P(Y>5) {m}") for m in ["persistence", "LR", "RF"]])
             .toPandas())
storm.to_csv(C.RESULTS / "test_case_study.csv", index=False)
print(peak.station_id, peak.ts, f"{peak.y_mm} mm in 30 minutes")
storm.iloc[::2].set_index("time").round(3)

S90 2024-12-29 08:40:00 65.0 mm in 30 minutes


,past30_mm,y_mm,P(Y>0) persistence,P(Y>0) LR,P(Y>0) RF,P(Y>5) persistence,P(Y>5) LR,P(Y>5) RF
time,,,,,,,,
2024-12-29 14:40,0.0,0.0,0.024,0.304,0.267,0.003,0.043,0.037
2024-12-29 14:50,0.0,1.2,0.024,0.079,0.145,0.003,0.010,0.019
2024-12-29 15:00,0.0,13.4,0.024,0.109,0.123,0.003,0.017,0.015
2024-12-29 15:10,0.0,26.0,0.024,0.236,0.238,0.003,0.053,0.045
2024-12-29 15:20,1.2,28.2,0.792,0.991,0.983,0.073,0.542,0.521
2024-12-29 15:30,13.4,17.8,0.932,1.000,1.000,0.390,0.944,0.904
2024-12-29 15:40,26.0,5.8,0.980,0.999,0.992,0.584,0.542,0.579
2024-12-29 15:50,28.2,5.8,0.980,1.000,0.996,0.584,0.685,0.552
2024-12-29 16:00,17.8,17.0,0.932,0.975,0.959,0.390,0.298,0.288


**Which tiers the final models rely on**

On a 5 % sample of 2024, the columns of one tier are shuffled together across rows, which breaks their link to the outcome while keeping their distribution, and the rise in RPS is measured. Unlike the impurity importance of the forest, this works the same way for both models and refits nothing.

In [12]:
sample = (spark.read.parquet(str(C.ASSEMBLED / "test")).sample(fraction=0.05, seed=C.SEED)
               .select("label", "w", "y_mm", *FEATURES).toPandas())


def sample_rps(model, pdf):
    return E.brier_rps(E.add_exceedance(model.transform(spark.createDataFrame(pdf))))[1]


base = {kind: sample_rps(model, sample) for kind, model in final.items()}
rows = []
for tier in TIERS:
    cols = [c for c in FEATURES if c.startswith(f"{tier}_")]
    shuffled = sample.copy()
    shuffled[cols] = sample[cols].to_numpy()[rng.permutation(len(sample))]
    rows.append({"tier shuffled": tier.upper(), "columns": len(cols),
                 **{f"RPS rise {kind}": sample_rps(model, shuffled) - base[kind] for kind, model in final.items()}})
reliance = pd.DataFrame(rows).set_index("tier shuffled")
reliance.to_csv(C.RESULTS / "test_tier_permutation.csv")
print("sample RPS:", {k: round(v, 5) for k, v in base.items()})
reliance.round(5)

sample RPS: {'LR': 0.09218, 'RF': 0.08863}


,columns,RPS rise LR,RPS rise RF
tier shuffled,,,
T0,15,0.00054,0.00046
T1,11,0.03022,0.05653
T2,15,0.08553,0.06336
T3,17,0.00393,0.00532
